# LLM Intention Probing, Honesty, Deception, and Honest Mistakes, Algoverse 2026 Spring, KMSA & Tommy
## Part 1: Preparation

In [32]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters
from utils.settings import *

# Utils
from utils.knowledge_check import (
    knowledge_check_truthfulqa, knowledge_check_mmlu,
    run_knowledge_check_truthfulqa, run_knowledge_check_mmlu,
)
from utils.generation import (
    generate_response,
    run_factual_generation, run_scenario_generation,
)
from utils.judge import (
    build_batch_requests_anthropic, parse_batch_results_anthropic,
    build_batch_requests_openai, parse_batch_results_openai,
    run_judge_anthropic, run_judge_openai,
    aggregate_judge_votes, build_full, print_threshold_summary,
)
from utils.activation import extract_activations, run_extract_activations, LABEL_MAP
from utils.analysis import (
    reduce_activations_pca, save_results_csv, select_pca_k, run_pca_reduction,
    filter_factual, build_probe_dataset,
)
from utils.probe import (
    probe_all_layers, probe_all_layers_binary,
    probe_all_layers_cascaded, probe_all_layers_mlp, probe_all_layers_cascaded_mlp,
)
from utils.plotting import (
    plot_macro_f1, plot_perclass_f1, plot_auroc, plot_top_confusion_matrices,
)

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Create output directories
for d in [
    KNOWLEDGE_TEST_DIR, RESPONSES_DIR,
    JUDGE_DIR, JUDGE_CLAUDE_HAIKU_DIR, JUDGE_GPT4O_MINI_DIR,
    OUTPUT_DIR, FIGURES_DIR,
    BINARY_DIR, TWAY_LR_DIR, TWAY_MLP_DIR, CASCADED_LR_DIR, CASCADED_MLP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Load fixed social scenario dataset
deception_df = pd.read_csv(DECEPTION_DATASET_PATH)
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts().to_string())

print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")

deception_dataset: (400, 6)
label
honest       200
deceptive    200

Device: cuda
GPU:  NVIDIA GeForce RTX 4090
VRAM: 25.4 GB
Model: Qwen/Qwen2.5-7B-Instruct


### 1.2 Load Model & Tokenizer

In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

N_LAYERS   = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded: Qwen/Qwen2.5-7B-Instruct
Layers: 28, hidden_dim: 3584


## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [34]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")

kc_tqa_df, tqa_passed_df, tqa_failed_df = run_knowledge_check_truthfulqa(
    tqa_mc, model, tokenizer, DEVICE, TRUTHFULQA_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(tqa_passed_df)} | Failed: {len(tqa_failed_df)}")

[skip] Already complete (817 rows): truthfulQA_test_results.csv

Passed: 398 | Failed: 419


### 2.2 MMLU

In [35]:
mmlu_mc = load_dataset("cais/mmlu", "all", split="test")

kc_mmlu_df, mmlu_passed_df, mmlu_failed_df = run_knowledge_check_mmlu(
    mmlu_mc, model, tokenizer, DEVICE, MMLU_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(mmlu_passed_df)} | Failed: {len(mmlu_failed_df)}")

Resuming: 14038 done, 0 remaining


MMLU knowledge check: 0it [00:00, ?it/s]

Done. Total: 14038 | Passed: 6596 | Failed: 7442

Passed: 6596 | Failed: 7442


## Part 3: Factual Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [36]:
tqa_resp_df = run_factual_generation(
    tqa_passed_df, tqa_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(tqa_resp_df["config"].value_counts().to_string())

[skip] Already complete (1215 rows): truthfulQA_responses.csv
config
B    419
A    398
C    398


#### 3.1.2 Claude Haiku Batch Judge

In [37]:
tqa_haiku_df = run_judge_anthropic(
    tqa_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_TQA_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_TQA_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

[skip] Already complete: judge_truthfulQA.csv


#### 3.1.3 GPT-4o-mini Batch Judge

In [38]:
tqa_gpt_df = run_judge_openai(
    tqa_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_TQA_PATH,
    state_path=JUDGE_GPT4O_MINI_TQA_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

[skip] Already complete: judge_truthfulQA.csv


### 3.2 MMLU Response Generation and Result Judge
#### 3.2.1 Response Generation

In [39]:
mmlu_resp_df = run_factual_generation(
    mmlu_passed_df, mmlu_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    MMLU_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(mmlu_resp_df["config"].value_counts().to_string())

[skip] Already complete (20634 rows): mmlu_responses.csv
config
B    7442
A    6596
C    6596


#### 3.2.2 Claude Haiku Batch Judge

In [40]:
mmlu_haiku_df = run_judge_anthropic(
    mmlu_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_MMLU_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_MMLU_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

[skip] Already complete: judge_mmlu.csv


#### 3.2.3 GPT-4o-mini Batch Judge

In [41]:
mmlu_gpt_df = run_judge_openai(
    mmlu_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_MMLU_PATH,
    state_path=JUDGE_GPT4O_MINI_MMLU_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

[skip] Already complete: judge_mmlu.csv


### 3.3 Judge Result Compile

In [42]:
if TRUTHFULQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
    tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
    mmlu_full = pd.read_csv(MMLU_FULL_PATH)
    print(f"Loaded tqa_full  ({len(tqa_full)} rows)")
    print(f"Loaded mmlu_full ({len(mmlu_full)} rows)")
else:
    tqa_votes  = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_TQA_PATH, JUDGE_GPT4O_MINI_TQA_PATH,
        vote_cols=VOTE_COLS,
    )
    mmlu_votes = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_MMLU_PATH, JUDGE_GPT4O_MINI_MMLU_PATH,
        vote_cols=VOTE_COLS,
    )
    tqa_full  = build_full(tqa_votes,  tqa_resp_df)
    mmlu_full = build_full(mmlu_votes, mmlu_resp_df)
    tqa_full.to_csv(TRUTHFULQA_FULL_PATH,  index=False)
    mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
    print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TRUTHFULQA_FULL_PATH.name}")
    print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH.name}")

print_threshold_summary(tqa_full,  "TruthfulQA")
print_threshold_summary(mmlu_full, "MMLU")

Loaded tqa_full  (1215 rows)
Loaded mmlu_full (22607 rows)

=== TruthfulQA — votes_correct distribution ===
config           A    B    C
votes_correct               
0               83  215  240
1                7   18   14
2               10   14    8
3               39   36   17
4               18   19    7
5               26   23   16
6              215   94   96

  Config A (truth)       — useful range: votes_correct ≥ 4
    >= 6/6 correct :   215 rows
    >= 5/6 correct :   241 rows
    >= 4/6 correct :   259 rows

  Config B (honest_mistake) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   215 rows
    <= 1/6 correct :   233 rows
    <= 2/6 correct :   247 rows

  Config C (deception) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   240 rows
    <= 1/6 correct :   254 rows
    <= 2/6 correct :   262 rows

=== MMLU — votes_correct distribution ===
config            A     B     C
votes_correct                  
0              2561  4090  3397
1               313   5

## Part 4: Scenario Response Generation

In [43]:
scenario_resp_df = run_scenario_generation(
    deception_df, model, tokenizer, DEVICE,
    SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

[skip] Already complete (200 pairs): scenario_responses.csv

Columns: ['pair_id', 'question', 'honest_scenario', 'honest_response', 'deceptive_scenario', 'deceptive_response']
   pair_id                                           question  \
0        0  Teacher: Why haven't you submitted your assign...   
1        1  Teacher: Why have you not submitted the scienc...   

                                     honest_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                     honest_response  \
0  I understand that this is a difficult situatio...   
1  Teacher: I understand that you're concerned ab...   

                                  deceptive_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                  deceptive_response  
0  I understand that this is a difficult situatio...  
1  Teacher:

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Probe Dataset

In [44]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, scenario_resp_df, PROBE_DATASET_PATH,
)

[skip] Loaded probe_dataset (11708 rows): probe_dataset.csv


### 5.2 Extract Activations

In [45]:
activations_arr, labels_arr = run_extract_activations(
    probe_dataset, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN, CHECKPOINT_EVERY,
)
print(f"\nLabel counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

Loading activations:   0%|          | 0/4699684992 [00:00<?, ?it/s]

Loading labels     :   0%|          | 0/93792 [00:00<?, ?it/s]

[local] activations (11708, 28, 3584), labels (11708,)

Label counts: {'truth': 3566, 'honest_mistake': 4305, 'deception': 3837}


## Part 6: Probe Training and Evaluation
### 6.1 Setup

In [46]:
labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

k_selection_df = select_pca_k(
    activations_arr, labels_str, PCA_K_VALUES, PCA_K_SELECTION_PATH,
)

[skip] Already complete (18 rows): pca_reduction_k_selection_results.csv


In [47]:
acts_reduced = run_pca_reduction(
    activations_arr, PCA_K,
    ACTIVATIONS_PCA_PATH, PCA_COMPONENTS_PATH, PCA_VARIANCE_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN,
)

[skip] Loaded reduced activations (11708, 28, 64): activations_pca64.npy


### 6.2 Baseline: Binary Classifier

In [48]:
results_binary_c1 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=1.0,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C1_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C1.pkl",
)
results_binary_c01 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=0.1,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C01_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C01.pkl",
)

[skip] probe_results_binary_pca64_C1.csv already exists (28 rows)
[skip] probe_results_binary_pca64_C01.csv already exists (28 rows)


In [49]:
plot_auroc(
    [(results_binary_c1, "C=1.0"), (results_binary_c01, "C=0.1")],
    BINARY_DIR / "figures" / "auroc.png",
    title="Binary Probe AUROC per Layer (truth vs deception)",
)

Saved auroc.png


### 6.3 Approach 1: Direct 3-Way LR Classifier

In [50]:
results_3way_lr = probe_all_layers(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=TWAY_LR_PATH,
    checkpoint_path=TWAY_LR_DIR / "checkpoint_3way_lr.pkl",
)

[skip] probe_results_3way_pca64.csv already exists (28 rows)


In [51]:
plot_macro_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "macro_f1.png", title="3-Way LR: Macro F1 per Layer")
plot_perclass_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "perclass_f1.png", title="3-Way LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_lr, TWAY_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.4 Approach 2: Direct 3-Way MLP Classifier

In [52]:
results_3way_mlp = probe_all_layers_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=TWAY_MLP_PATH,
    checkpoint_path=TWAY_MLP_DIR / "checkpoint_3way_mlp.pkl",
)

[skip] probe_results_3way_mlp_pca64.csv already exists (28 rows)


In [53]:
plot_macro_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "macro_f1.png", title="3-Way MLP: Macro F1 per Layer")
plot_perclass_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "perclass_f1.png", title="3-Way MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_mlp, TWAY_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="MLP ")
plot_macro_f1(
    [(results_3way_lr, "LR"), (results_3way_mlp, "MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_lr_vs_mlp.png",
    title="3-Way Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_lr_vs_mlp.png


### 6.5 Approach 3: 2-Stage LR Classifier

In [54]:
results_cascaded_lr = probe_all_layers_cascaded(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=CASCADED_LR_PATH,
    checkpoint_path=CASCADED_LR_DIR / "checkpoint_cascaded_lr.pkl",
)

[skip] probe_results_cascaded_lr.csv already exists (28 rows)


In [55]:
plot_macro_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "macro_f1.png", title="Cascaded LR: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "perclass_f1.png", title="Cascaded LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.6 Approach 4: 2-Stage MLP Classifier

In [56]:
results_cascaded_mlp = probe_all_layers_cascaded_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=CASCADED_MLP_PATH,
    checkpoint_path=CASCADED_MLP_DIR / "checkpoint_cascaded_mlp.pkl",
)

[skip] probe_results_cascaded_mlp.csv already exists (28 rows)


In [57]:
plot_macro_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "macro_f1.png", title="Cascaded MLP: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "perclass_f1.png", title="Cascaded MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded MLP ")
plot_macro_f1(
    [(results_cascaded_lr, "Cascaded LR"), (results_cascaded_mlp, "Cascaded MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_cascaded_lr_vs_mlp.png",
    title="Cascaded Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_cascaded_lr_vs_mlp.png


## Part 7: Model Comparison

In [58]:
# Load all probe results from CSV (safe to run after kernel restart)
r_lr    = pd.read_csv(TWAY_LR_PATH)
r_mlp   = pd.read_csv(TWAY_MLP_PATH)
r_clr   = pd.read_csv(CASCADED_LR_PATH)
r_cmlp  = pd.read_csv(CASCADED_MLP_PATH)
r_bin1  = pd.read_csv(BINARY_C1_PATH)
r_bin01 = pd.read_csv(BINARY_C01_PATH)

PROBE_RESULTS = [
    (r_lr,   "3-Way LR"),
    (r_mlp,  "3-Way MLP"),
    (r_clr,  "Cascaded LR"),
    (r_cmlp, "Cascaded MLP"),
]
SUMMARY_DIR = OUTPUT_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Loaded all probe results.")

Loaded all probe results.


In [59]:
layers = r_lr["layer"].values

# ── Table 1: Macro F1 per layer ───────────────────────────────────────────────
t1 = pd.DataFrame({"layer": layers})
for df, name in PROBE_RESULTS:
    t1[name] = df["f1_macro"].values
t1.to_csv(SUMMARY_DIR / "summary_macro_f1.csv", index=False)
print(t1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
for df, name in PROBE_RESULTS:
    ax.plot(layers, df["f1_macro"], marker="o", markersize=3, label=name)
ax.set_xlabel("Layer"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 per Layer — All Probes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "macro_f1_all_probes.png", dpi=150); plt.close(fig)
print("Saved macro_f1_all_probes.png")

# ── Tables 2/3/4: Per-class F1 per layer ─────────────────────────────────────
for cls in ["truth", "honest_mistake", "deception"]:
    t = pd.DataFrame({"layer": layers})
    for df, name in PROBE_RESULTS:
        t[name] = df[f"f1_{cls}"].values
    t.to_csv(SUMMARY_DIR / f"summary_f1_{cls}.csv", index=False)
    print(f"\n── {cls} ──")
    print(t.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    for df, name in PROBE_RESULTS:
        ax.plot(layers, df[f"f1_{cls}"], marker="o", markersize=3, label=name)
    ax.set_xlabel("Layer"); ax.set_ylabel(f"F1 ({cls})")
    ax.set_title(f"{cls} F1 per Layer — All Probes")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(SUMMARY_DIR / f"f1_{cls}_all_probes.png", dpi=150); plt.close(fig)
    print(f"Saved f1_{cls}_all_probes.png")

# ── Table 5: AUROC per layer (binary baseline) ────────────────────────────────
t5 = pd.DataFrame({
    "layer":       r_bin1["layer"].values,
    "Binary C=1.0": r_bin1["auroc"].values,
    "Binary C=0.1": r_bin01["auroc"].values,
})
t5.to_csv(SUMMARY_DIR / "summary_auroc_binary.csv", index=False)
print("\n── Binary AUROC ──")
print(t5.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r_bin1["layer"], r_bin1["auroc"], marker="o", markersize=3, label="C=1.0")
ax.plot(r_bin01["layer"], r_bin01["auroc"], marker="o", markersize=3, label="C=0.1")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Binary Probe AUROC per Layer (truth vs deception)")
ax.set_ylim(0.5, 1.0); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "auroc_binary.png", dpi=150); plt.close(fig)
print("Saved auroc_binary.png")

 layer  3-Way LR  3-Way MLP  Cascaded LR  Cascaded MLP
     0  0.666510   0.701759     0.668568      0.703276
     1  0.674338   0.698381     0.677770      0.702848
     2  0.683888   0.710255     0.680560      0.716039
     3  0.686535   0.690261     0.685653      0.700841
     4  0.702835   0.708932     0.697592      0.716359
     5  0.707831   0.724491     0.707643      0.727781
     6  0.707740   0.720436     0.704107      0.729390
     7  0.707548   0.718157     0.711329      0.721399
     8  0.714797   0.727447     0.715563      0.735384
     9  0.729323   0.736139     0.727762      0.743577
    10  0.734645   0.736634     0.735110      0.744940
    11  0.742745   0.748946     0.740460      0.752655
    12  0.753476   0.756737     0.752083      0.763262
    13  0.761778   0.757147     0.760828      0.766185
    14  0.769659   0.768992     0.767125      0.777499
    15  0.786851   0.780602     0.788961      0.790950
    16  0.793331   0.782659     0.789442      0.789597
    17  0.